# Reward Log Viewer

This notebook loads reward logs written by `zdt1_eva.py`, `nsga-eic.py`, and `nsga-nda.py` from the `reward_logs/` folder and plots them.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
LOG_DIR = ROOT / 'reward_logs'

def load_log(name):
    path = LOG_DIR / name
    if not path.exists():
        print(f'Missing: {path}')
        return None
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    print(f'Loaded: {path}')
    return data

zdt1_eva_log = load_log('zdt1_eva_train_rewards.json')
nsga_eic_log = load_log('nsga_eic_zdt1_rewards.json')
nsga_eic_deepic_log = load_log('nsga_eic_deepic_zdt1_rewards.json')
nsga_nda_log = load_log('nsga_nda_zdt1_rewards.json')

In [ ]:
if zdt1_eva_log is not None:
    epoch_rewards = zdt1_eva_log.get('epoch_mean_rewards', [])
    plt.figure(figsize=(8, 4))
    plt.plot(epoch_rewards, marker='o')
    plt.title('zdt1_eva.py Training Reward by Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Reward')
    plt.grid(True)
    plt.show()

    raw_rewards = [entry['reward'] for entry in zdt1_eva_log.get('records', [])]
    if raw_rewards:
        window = min(100, len(raw_rewards))
        kernel = np.ones(window) / window
        smooth = np.convolve(raw_rewards, kernel, mode='valid')
        plt.figure(figsize=(10, 4))
        plt.plot(raw_rewards, alpha=0.25, label='Raw Reward')
        plt.plot(range(window - 1, window - 1 + len(smooth)), smooth, linewidth=2, label=f'Rolling Mean ({window})')
        plt.title('zdt1_eva.py Step Reward History')
        plt.xlabel('Training Step')
        plt.ylabel('Reward')
        plt.grid(True)
        plt.legend()
        plt.show()

In [ ]:
plt.figure(figsize=(9, 4))
plotted = False

for label, payload in [
    ('NSGA-EIC', nsga_eic_log),
    ('DeepIC in nsga-eic.py', nsga_eic_deepic_log),
    ('NSGA-NDA', nsga_nda_log),
]:
    if payload is None:
        continue
    rewards = payload.get('reward_history', [])
    if not rewards:
        continue
    plt.plot(rewards, marker='o', label=label)
    plotted = True

if plotted:
    plt.title('Reward History Across ZDT1 Runs')
    plt.xlabel('Outer Iteration')
    plt.ylabel('Reward')
    plt.grid(True)
    plt.legend()
    plt.show()
else:
    print('No run reward logs available yet.')